<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/4-query-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara Query API Examples

In this notebook we demonstrate how to use Vectara's Query API using direct REST API calls. We'll cover:
- Single corpus queries with hybrid search and reranking
- Multiple corpora queries
- Metadata filtering
- Streaming responses
- Custom prompts with `prompt_template`

## About Vectara

[Vectara](https://vectara.com/) is an Agent Platform for trusted enterprise AI — a unified Agentic RAG platform with built-in retrieval, orchestration, and governance. See [Notebook 1](1-corpus-creation.ipynb) for the full overview of features and deployment options.

## Getting Started

This notebook assumes you've completed Notebooks 1 and 2:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs) with Boomerang embeddings
- Notebook 2: Ingested AI research papers and Vectara documentation


In [1]:
import os
import requests
import json

# Set up authentication
api_key = os.environ['VECTARA_API_KEY']

# Get corpus keys from environment (set these from Notebook 1 output)
research_corpus_key = 'tutorial-ai-research-papers'
docs_corpus_key = 'tutorial-vectara-docs'

# Base URL for Vectara API v2
BASE_URL = "https://api.vectara.io/v2"

# Common headers for all requests
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'x-api-key': api_key
}

print(f"Research Corpus: {research_corpus_key}")
print(f"Docs Corpus: {docs_corpus_key}")

Research Corpus: tutorial-ai-research-papers
Docs Corpus: tutorial-vectara-docs


## Example 1: Basic Query with Hybrid Search and Reranking

This example demonstrates a single corpus query using:
- Hybrid search (lexical_interpolation=0.005 for best semantic search)
- Chain reranker combining the `qwen3-reranker` with MMR (diversity_bias=0.05) for improved relevance and diversity
- Multi-stage retrieval: retrieve 100 results, keep the top 30 after reranking, and send the top 10 to the LLM for generation
- Generation with vectara-summary-ext-24-05-med-omni preset
- Factual Consistency Score to detect potential hallucinations

> Reranker availability depends on your account. These examples reference the reranker by `reranker_name` (`qwen3-reranker`); list the rerankers available to you with `GET /v2/rerankers`. Other options include `Rerank_Multilingual_v1` (the Vectara Multilingual Reranker).

In [2]:
# Construct the query request - querying research papers corpus
query_request = {
    "query": "What is retrieval augmented generation?",
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 30,
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ],
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

# Make the query request
url = f"{BASE_URL}/query"
response = requests.post(url, headers=headers, json=query_request)

if response.status_code == 200:
    result = response.json()
    print("\n=== Generated Summary ===")
    print(result['summary'])
    print(f"\n=== Factual Consistency Score: {result.get('factual_consistency_score', 'N/A')} ===")
else:
    print(f"Error: {response.status_code}")
    print(response.text)


=== Generated Summary ===
Retrieval augmented generation (RAG) is a method that enhances language generation models by integrating both parametric and non-parametric memory components. In RAG models, the parametric memory is typically a pre-trained sequence-to-sequence (seq2seq) transformer model, while the non-parametric memory consists of a dense vector index, such as a database of Wikipedia articles. This index is accessed using a pre-trained neural retriever. The combination of these components allows the model to generate more factual and specific outputs by retrieving relevant information from the non-parametric memory during the generation process. RAG models are trained end-to-end and can be applied to various knowledge-intensive NLP tasks [1], [2].

=== Factual Consistency Score: 0.96484375 ===


### Examining Search Results and Citations

The response includes the retrieved documents that were used to generate the summary, along with citation information.

In [3]:
if response.status_code == 200:
    result = response.json()
    
    # Display first 5 search results
    print("\n=== Top Search Results ===")
    for i, search_result in enumerate(result.get('search_results', [])[:5], 1):
        print(f"\n--- Result {i} ---")
        print(f"Text: {search_result['text'][:200]}...")
        print(f"Score: {search_result.get('score', 'N/A')}")
        print(f"Document ID: {search_result.get('document_id', 'N/A')}")
        if 'document_metadata' in search_result:
            print(f"Metadata: {search_result['document_metadata']}")


=== Top Search Results ===

--- Result 1 ---
Text: but have only explored open-domain extractive question answering. Here, we bring hybrid parametric
and non-parametric memory to the “workhorse of NLP,” i.e. sequence-to-sequence (seq2seq) models. We e...
Score: 0.983217716217041
Document ID: rag-retrieval-augmented-generation.pdf
Metadata: {'PTEX.Fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'CreationDate': 'D:20210413004838Z', 'Keywords': '', 'Producer': 'pdfTeX-1.40.21', 'Author': '', 'Title': '', 'Creator': 'LaTeX with hyperref', 'ModDate': 'D:20210413004838Z', 'Trapped': '/False', 'Subject': '', 'source': 'arxiv', 'year': 2020, 'topic': 'RAG', 'title': 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks', 'authors': 'Lewis et al.'}

--- Result 2 ---
Text: Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentia

## Example 2: Querying Multiple Corpora

Vectara allows you to query across multiple corpora simultaneously. This is useful when you have data organized across different collections.

In [4]:
# Query both corpora simultaneously
# This combines results from research papers AND documentation
multi_corpus_request = {
    "query": "How do Agents work with Vectara?",
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            },
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 30
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ],
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

response = requests.post(f"{BASE_URL}/query", headers=headers, json=multi_corpus_request)

if response.status_code == 200:
    result = response.json()
    print("\n=== Generated Summary (Multiple Corpora) ===")
    print(result['summary'])
    print(f"\n=== Factual Consistency Score: {result.get('factual_consistency_score', 'N/A')} ===")
    
    # Show which corpus each result came from
    print("\n=== Result Sources ===")
    for i, search_result in enumerate(result.get('search_results', [])[:5], 1):
        doc_meta = search_result.get('document_metadata', {})
        source = doc_meta.get('source', 'unknown')
        title = doc_meta.get('title', 'N/A')
        print(f"{i}. Source: {source}, Title: {title}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)


=== Generated Summary (Multiple Corpora) ===
Agents in Vectara work by utilizing an API-first platform that allows for the creation and orchestration of AI agents and assistants. These agents can retrieve knowledge, reason over context, use tools, and deliver grounded answers with detailed audit trails and access controls. Vectara supports various deployment environments, including SaaS, VPC, and on-premises, ensuring data security and compliance. Agents interact with external systems through tools and connectors, enabling them to perform tasks like Salesforce lookups or SQL queries, and integrate with platforms like Slack, Google Chat, and Zoom Contact Center. The platform provides APIs for agent creation, session management, and data retrieval, allowing agents to maintain context across interactions and execute workflows that align with business processes [1], [2], [4], [6], [7].

=== Factual Consistency Score: 1.0 ===

=== Result Sources ===
1. Source: vectara_docs, Title: Understa

## Example 3: Metadata Filtering

You can filter search results using metadata filters. This allows you to narrow down results based on document or chunk-level metadata.

In [5]:
# Example with metadata filtering
# Filter to only get research papers from 2021 or later
filtered_request = {
    "query": "How are dense retrieval models trained and evaluated?",
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005,
                # Filter for recent papers
                "metadata_filter": "doc.year >= 2021"
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 30
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ],
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

response = requests.post(f"{BASE_URL}/query", headers=headers, json=filtered_request)

if response.status_code == 200:
    result = response.json()
    print("\n=== Generated Summary (With Metadata Filter) ===")
    print(result['summary'])
    print(f"\n=== Number of results: {len(result.get('search_results', []))} ===")
    
    # Show filtered results
    print("\n=== Filtered Papers ===")
    for search_result in result.get('search_results', [])[:3]:
        doc_meta = search_result.get('document_metadata', {})
        print(f"- {doc_meta.get('title', 'N/A')} ({doc_meta.get('year', 'N/A')}) - Topic: {doc_meta.get('topic', 'N/A')}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)


=== Generated Summary (With Metadata Filter) ===
Dense retrieval models are trained using techniques such as dual-encoder architectures, where queries and documents are independently mapped to vector spaces. Training often involves using a similarity function within an embedding space, with methods like InfoNCE loss, which utilizes in-batch negatives for each query [2]. Additionally, models like DPR are trained with hard negatives and in-batch negatives across multiple datasets [3]. Evaluation of these models is typically conducted on benchmark datasets such as TREC-DL and MSMARCO, focusing on metrics like recall quality and NDCG@10 to compare performance against traditional methods like BM25 [1], [4].

=== Number of results: 30 ===

=== Filtered Papers ===
- Efficiently Teaching an Effective Dense Retriever with Balanced Topic Aware Sampling (2021) - Topic: retrieval
- BEIR: A Heterogenous Benchmark for Zero-shot Evaluation of Information Retrieval Models (2021) - Topic: retrieval
- 

## Example 4: Streaming Responses

For better user experience, you can stream the generated response in real-time using Server-Sent Events (SSE).

In [6]:
# Streaming query request - query the documentation corpus
streaming_request = {
    "query": "How do I use chunking with Vectara",
    "stream_response": True,
    "search": {
        "corpora": [
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 30
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ],
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 15,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

# Make streaming request
streaming_headers = headers.copy()
streaming_headers['Accept'] = 'text/event-stream'

response = requests.post(
    f"{BASE_URL}/query",
    headers=streaming_headers,
    json=streaming_request,
    stream=True
)

print("\n=== Streaming Response ===")
if response.status_code == 200:
    for line in response.iter_lines():
        if line:
            line_str = line.decode('utf-8')
            if line_str.startswith('data:'):
                try:
                    data = json.loads(line_str[5:])  # Remove 'data: ' prefix
                    # Handle different event types
                    if data.get('type') == 'generation_chunk':
                        # Print generation text as it arrives
                        print(data.get('generation_chunk', ''), end='', flush=True)
                    elif data.get('type') == 'factual_consistency_score':
                        print(f"\n\n=== FCS: {data.get('factual_consistency_score')} ===")
                    elif data.get('type') == 'search_results':
                        # Search results arrive before generation starts
                        pass
                    elif data.get('type') == 'error':
                        # The stream can fail after it starts (e.g. LLM quota exceeded)
                        print(f"\n\n=== Stream error: {data.get('messages')} ===")
                except json.JSONDecodeError:
                    pass
    print("\n")
else:
    print(f"Error: {response.status_code}")
    print(response.text)


=== Streaming Response ===


To 

use c

hun

kin

...

a

tegy

[1],

[2], [5].



=== FCS: 0.88671875 ===




## Example 5: Custom Prompts with prompt_template

Vectara's Prompt Engine allows you to customize the LLM behavior beyond the default presets by using the `prompt_template` parameter. You can use Velocity Template syntax to access variables like:
- `$vectaraQuery` - The user's query text
- `$vectaraQueryResults` - Array of retrieved search results
- `$vectaraLangCode` - ISO639 language code (e.g., "eng")
- `$vectaraOutChars` - Character limit for output

This example demonstrates how to create a custom prompt that formats responses in a specific way.

In [7]:
# Custom prompt template using Velocity syntax
# This example creates a structured response with sections and bullet points
custom_prompt_template = [
    {
        "role": "system",
        "content": """You are a helpful AI assistant that provides clear, well-structured answers. 
Your responses should be formatted with the following sections:
1. A brief summary (2-3 sentences)
2. Key points as bullet points

Use the search results provided to ground your answer in facts."""
    },
    {
        "role": "user", 
        "content": """Question: $vectaraQuery

Retrieved Context:
#foreach ($qResult in $vectaraQueryResults)
- $qResult.text()
#end

Please provide a well-structured answer to the question above using the retrieved context. Format your response with clear sections as instructed."""
    }
]

# Query request with custom prompt
custom_prompt_request = {
    "query": "How can I improve retrieval quality in RAG systems?",
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 30
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ],
        }
    },
    "generation": {
        "prompt_template": json.dumps(custom_prompt_template),
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    },
    "save_history": True
}

response = requests.post(f"{BASE_URL}/query", headers=headers, json=custom_prompt_request)

if response.status_code == 200:
    result = response.json()
    print("\n=== Generated Response with Custom Prompt ===")
    print(result['summary'])
    print(f"\n=== Factual Consistency Score: {result.get('factual_consistency_score', 'N/A')} ===")
else:
    print(f"Error: {response.status_code}")
    print(response.text)


=== Generated Response with Custom Prompt ===
**Summary:**
Improving retrieval quality in Retrieval-Augmented Generation (RAG) systems involves optimizing the retriever component to ensure it effectively retrieves relevant information for the task at hand. Key strategies include using learned retrieval mechanisms, adjusting the number of documents retrieved, and leveraging flexible retrieval indices that can be updated without retraining.

**Key Points:**

- **Learned Retrieval Mechanisms:**
  - RAG systems benefit from learned retrieval mechanisms, which improve results across various tasks compared to fixed retrieval systems like BM25.
  - The retriever in RAG is initialized using the Dense Passage Retrieval (DPR) system, which is trained with retrieval supervision on datasets like Natural Questions and TriviaQA.

- **Comparison with Other Systems:**
  - RAG's dense retriever outperforms traditional word overlap-based systems such as BM25, except in specific cases like the FEVER dat

### Advanced Custom Prompt: Accessing Metadata

You can also access document and part metadata within your custom prompts using:
- `$qResult.docMetadata()` - Access document-level metadata
- `$qResult.partMetadata()` - Access chunk-level metadata
- `.get('fieldname')` - Retrieve specific metadata fields

This example shows how to create a prompt that includes citation information with metadata.

In [8]:
# Advanced custom prompt with metadata access
# This prompt includes paper titles and authors in the context
advanced_prompt_template = [
    {
        "role": "system",
        "content": """You are a research assistant helping to synthesize information from academic papers. 
Provide a comprehensive answer citing specific papers by their title and authors."""
    },
    {
        "role": "user",
        "content": """Question: $vectaraQuery

Research Papers Context:
#foreach ($qResult in $vectaraQueryResults)
---
Title: $qResult.docMetadata().get('title')
Topic: $qResult.docMetadata().get('topic')
Content: $qResult.text()

#end

Based on the research papers above, provide a detailed answer that cites specific papers by title."""
    }
]

# Query with advanced custom prompt
advanced_prompt_request = {
    "query": "What are the challenges with RAG systems?",
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 30
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ],
        }
    },
    "generation": {
        "prompt_template": json.dumps(advanced_prompt_template),
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 7,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    },
    "save_history": True,
}

response = requests.post(f"{BASE_URL}/query", headers=headers, json=advanced_prompt_request)

if response.status_code == 200:
    result = response.json()
    print("\n=== Response with Metadata-Rich Custom Prompt ===")
    print(result['summary'])
    print(f"\n=== Factual Consistency Score: {result.get('factual_consistency_score', 'N/A')} ===")
else:
    print(f"Error: {response.status_code}")
    print(response.text)


=== Response with Metadata-Rich Custom Prompt ===
Retrieval-Augmented Generation (RAG) systems, while offering significant advantages in terms of grounding language models in factual knowledge, face several challenges as highlighted in the research papers.

1. **Factuality and Bias**: One of the primary challenges with RAG systems is the reliance on external knowledge sources like Wikipedia, which may not always be entirely factual or free from bias. This can lead to the generation of misleading or biased content. The paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" by the authors notes that similar concerns as those for GPT-2 are applicable to RAG, including the potential for generating abusive, fake, or misleading content, impersonating others, or automating spam/phishing content (Title: "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks").

2. **Provenance and Knowledge Updating**: Another challenge is providing provenance for the decisions ma